In [ ]:
# =========================================================
# STEP 1 — MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================================================
# STEP 2 — INSTALL REQUIRED LIBRARIES
# =========================================================

!pip install -q pdf2image tensorflow
!apt-get install -y poppler-utils

In [ ]:
# =========================================================
# STEP 3 — IMPORT LIBRARIES
# =========================================================

import os
import random
import shutil
import numpy as np
import tensorflow as tf

from PIL import Image
from pdf2image import convert_from_path

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout

print('Libraries imported successfully!')

In [ ]:
# =========================================================
# STEP 4 — CONVERT PDF RESUMES TO IMAGES
# =========================================================
# INPUT:  Folder containing all resume PDFs
# OUTPUT: Folder containing converted JPG images
# =========================================================

# CHANGE ONLY THESE TWO PATHS
pdf_folder    = "/content/drive/MyDrive/Classroom/AI 6c 4(Lab)/Resume folder"
output_folder = "/content/drive/MyDrive/Classroom/AI 6c 4(Lab)/Resume images"

os.makedirs(output_folder, exist_ok=True)

pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith('.pdf')]

print('Total PDFs Found:', len(pdf_files))

saved_count = 0

for pdf_file in pdf_files:
    try:
        pdf_path = os.path.join(pdf_folder, pdf_file)
        pages    = convert_from_path(pdf_path, first_page=1, last_page=1)
        image    = pages[0]

        image_name = pdf_file.replace('.pdf', '.jpg')
        save_path  = os.path.join(output_folder, image_name)

        image.save(save_path, 'JPEG')
        saved_count += 1

    except Exception as e:
        print('Error converting:', pdf_file, '->', e)

print('Total Images Saved:', saved_count)

In [ ]:
# =========================================================
# STEP 5 — DISTRIBUTE IMAGES INTO CLASS FOLDERS
# =========================================================
# Randomly assigns each resume image to Good / Average / Poor
# NOTE: For better accuracy, manually label your images later
# =========================================================

source_folder = "/content/drive/MyDrive/Classroom/AI 6c 4(Lab)/Resume images"
output_folder = "/content/resume_images"

classes = ['Good', 'Average', 'Poor']

# Create class folders
for cls in classes:
    os.makedirs(os.path.join(output_folder, cls), exist_ok=True)

# Get all images
images = [f for f in os.listdir(source_folder)
          if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print('Total images found:', len(images))

# Distribute randomly
for img in images:
    label    = random.choice(classes)
    src_path = os.path.join(source_folder, img)
    dst_path = os.path.join(output_folder, label, img)
    shutil.copy(src_path, dst_path)

# Verify distribution
print('\nDistribution:')
for cls in classes:
    count = len(os.listdir(os.path.join(output_folder, cls)))
    print(f'  {cls}: {count} images')

In [ ]:
# =========================================================
# STEP 6 — TRAIN THE MODEL
# =========================================================
# Model: MobileNetV2 (Transfer Learning)
# Classes: Good / Average / Poor
# =========================================================

dataset_path = "/content/resume_images"

# Load dataset
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=8,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=8,
    class_mode='categorical',
    subset='validation',
    shuffle=True
)

print('Train Samples   :', train_data.samples)
print('Val Samples     :', val_data.samples)
print('Classes         :', train_data.class_indices)

if train_data.samples == 0:
    raise ValueError('Training data is empty. Check folder structure!')

# Build model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = Sequential([
    base_model,
    Flatten(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history = model.fit(train_data, validation_data=val_data, epochs=10)

# Save model
model.save('/content/resume_quality_model.h5')

print('\nModel saved to: /content/resume_quality_model.h5')

In [ ]:
# =========================================================
# STEP 7 — PREDICT RESUME QUALITY
# =========================================================
# Change test_image_path to your resume image
# =========================================================

from tensorflow.keras.preprocessing import image as keras_image

# CHANGE THIS PATH
test_image_path = "/content/drive/MyDrive/Classroom/AI 6c 4(Lab)/dummy_resume_pdfs/resume pic.png"

img       = keras_image.load_img(test_image_path, target_size=(224, 224))
img_array = keras_image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

classes        = ['Average', 'Good', 'Poor']
predicted_class = classes[np.argmax(prediction)]
confidence      = round(np.max(prediction) * 100, 2)

print('=================================')
print('Prediction :', predicted_class)
print('Confidence :', confidence, '%')
print('=================================')